# Лабораторна робота 2. Завдання 6–10 (AID, варіант 3)

* **6** — компактна CNN-учень, навчена з нуля за мітками датасету;
* **7** — такий самий учень із **дистиляцією знань** від кращої за validation моделі (завдання 3–5);
* **8** — аналіз помилок учителя та учнів: матриці помилок, неправильні прогнози, **Grad-CAM**;
* **9** — які класи покращились / погіршились в експериментах 4 (аугментація) і 7 (дистиляція), виправлені та нові помилки;
* **10** — зведена таблиця: точність, компактність, **стійкість** до спотворень; висновки.

Дані, поділ (15 val + 20 test з кожного класу), кешування, аугментація — ті самі, що в попередніх ноутбуках.
Ваги exp3–exp5 беруться з виходу кернелів `lab2-aid-steps3-4` і `lab2-aid-step5-finetune`
(на Kaggle підключені як input, локально — `steps3_4/out`, `step5/out`).

## 0. Налаштування

In [ ]:
import os, io, json, time, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models
from torchvision.transforms.v2 import functional as TF
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
IMG_SIZE = 224                 # стандартний вхід для моделей, навчених на ImageNet
BATCH_SIZE = 64
EPOCHS_ST = 40                 # бюджет навчання учня — однаковий для exp6 та exp7
LR_ST = 3e-3                   # max LR для OneCycle (учень навчається з нуля)
WEIGHT_DECAY = 5e-4
KD_CONFIGS = [(4.0, 0.7)]      # (температура T, вага дистиляційної складової α); можна додати, напр.,
                               # (2.0, 0.7), (8.0, 0.7), (4.0, 0.5), (4.0, 0.9) — по ~10 хв на T4 кожен,
                               # кращий обирається за validation
BLUR_SIGMA = (0.5, 1.2)        # «легке» розмиття: sigma гаусового ядра 5×5
NUM_WORKERS = min(4, os.cpu_count())
MP_CONTEXT = 'fork' if NUM_WORKERS > 0 else None   # воркери бачать кеш зображень у RAM без копіювання

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('device:', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
print('outputs ->', OUT_DIR)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

## 1. Дані та поділ (як у попередніх завданнях)

In [ ]:
VAL_TEST = {   # клас: ([номери файлів val], [номери файлів test])
    'Airport': ([20, 22, 23, 54, 125, 154, 202, 236, 238, 244, 256, 294, 297, 306, 309],
               [1, 29, 38, 46, 47, 85, 91, 97, 190, 206, 223, 231, 234, 251, 262, 278, 311, 314, 329, 345]),
    'BareLand': ([57, 58, 73, 96, 98, 104, 112, 154, 181, 205, 232, 236, 245, 273, 303],
                [2, 5, 40, 60, 62, 63, 77, 109, 115, 120, 146, 163, 167, 187, 192, 217, 221, 231, 267, 282]),
    'BaseballField': ([3, 13, 14, 36, 74, 86, 98, 132, 135, 145, 149, 153, 180, 183, 197],
                     [10, 26, 40, 53, 55, 58, 80, 119, 129, 131, 147, 173, 175, 178, 187, 188, 198, 205, 211, 217]),
    'Beach': ([11, 28, 43, 93, 107, 126, 136, 180, 269, 274, 286, 293, 321, 341, 381],
             [3, 64, 72, 77, 128, 134, 142, 153, 161, 171, 175, 246, 275, 279, 280, 292, 310, 313, 316, 333]),
    'Bridge': ([64, 66, 90, 124, 135, 141, 161, 168, 207, 210, 211, 216, 225, 275, 350],
              [33, 35, 40, 49, 52, 55, 74, 139, 150, 151, 159, 181, 192, 198, 247, 256, 301, 338, 340, 352]),
    'Center': ([35, 47, 55, 62, 79, 83, 156, 158, 170, 171, 208, 222, 228, 239, 240],
              [11, 14, 21, 37, 53, 61, 103, 106, 164, 168, 174, 182, 202, 204, 214, 218, 223, 232, 235, 251]),
    'Church': ([26, 47, 49, 71, 80, 82, 97, 114, 117, 125, 160, 187, 203, 209, 233],
              [3, 7, 9, 20, 28, 51, 53, 58, 63, 70, 74, 78, 99, 151, 175, 197, 199, 234, 237, 240]),
    'Commercial': ([9, 29, 106, 116, 133, 168, 182, 211, 218, 231, 282, 287, 318, 349, 350],
                  [2, 21, 33, 87, 122, 124, 130, 132, 140, 162, 185, 201, 223, 235, 255, 273, 307, 322, 332, 342]),
    'DenseResidential': ([8, 19, 29, 41, 66, 88, 153, 172, 263, 265, 275, 318, 332, 365, 399],
                        [10, 46, 54, 57, 60, 63, 68, 76, 83, 100, 110, 111, 113, 188, 190, 197, 249, 259, 294, 335]),
    'Desert': ([12, 35, 71, 98, 126, 202, 223, 237, 243, 244, 270, 283, 290, 293, 299],
              [23, 44, 54, 64, 69, 73, 90, 121, 138, 144, 152, 185, 186, 231, 255, 256, 266, 279, 289, 292]),
    'Farmland': ([86, 114, 123, 188, 192, 194, 199, 213, 221, 228, 232, 271, 341, 352, 355],
                [3, 21, 43, 100, 155, 195, 201, 208, 215, 217, 220, 226, 236, 253, 270, 273, 290, 320, 329, 336]),
    'Forest': ([3, 22, 37, 40, 70, 75, 112, 114, 141, 149, 197, 198, 221, 224, 235],
              [1, 4, 10, 23, 26, 29, 56, 58, 87, 101, 130, 131, 137, 161, 163, 168, 172, 185, 200, 213]),
    'Industrial': ([19, 23, 55, 56, 117, 128, 131, 138, 145, 151, 232, 279, 293, 350, 376],
                  [11, 13, 24, 25, 32, 48, 62, 64, 92, 123, 130, 143, 178, 217, 225, 228, 303, 308, 309, 330]),
    'Meadow': ([3, 15, 30, 32, 102, 115, 116, 136, 154, 166, 204, 208, 210, 217, 231],
              [2, 27, 45, 46, 49, 93, 105, 119, 122, 126, 129, 147, 193, 197, 198, 211, 241, 251, 257, 273]),
    'MediumResidential': ([4, 6, 14, 19, 37, 67, 75, 116, 119, 125, 180, 182, 188, 238, 249],
                         [1, 3, 5, 8, 11, 13, 15, 22, 24, 29, 109, 114, 130, 163, 202, 205, 207, 210, 284, 289]),
    'Mountain': ([6, 12, 22, 29, 40, 50, 70, 91, 117, 132, 145, 175, 206, 239, 310],
                [11, 16, 20, 24, 51, 54, 98, 101, 119, 137, 142, 146, 154, 166, 185, 190, 218, 225, 237, 336]),
    'Park': ([3, 7, 32, 39, 82, 126, 127, 176, 192, 249, 256, 269, 284, 304, 322],
            [1, 2, 33, 54, 57, 83, 105, 112, 129, 168, 184, 201, 225, 228, 288, 289, 310, 311, 321, 345]),
    'Parking': ([2, 109, 117, 140, 144, 165, 215, 273, 282, 284, 287, 302, 336, 341, 377],
               [7, 21, 94, 98, 110, 148, 167, 169, 184, 204, 210, 214, 219, 230, 257, 275, 290, 345, 354, 386]),
    'Playground': ([6, 32, 65, 90, 109, 133, 151, 174, 214, 222, 250, 255, 283, 299, 356],
                  [25, 41, 43, 53, 72, 75, 111, 112, 116, 122, 178, 202, 224, 280, 296, 304, 321, 343, 349, 364]),
    'Pond': ([58, 69, 121, 165, 174, 181, 182, 186, 193, 200, 223, 276, 331, 392, 419],
            [13, 16, 30, 102, 105, 145, 160, 167, 180, 204, 211, 220, 231, 251, 280, 324, 371, 382, 396, 418]),
    'Port': ([3, 22, 30, 38, 51, 63, 64, 94, 137, 158, 161, 168, 205, 264, 292],
            [75, 79, 92, 99, 104, 173, 195, 200, 220, 231, 235, 237, 251, 269, 298, 303, 309, 334, 340, 375]),
    'RailwayStation': ([18, 29, 36, 39, 56, 91, 94, 143, 149, 150, 151, 174, 194, 242, 250],
                      [6, 25, 41, 43, 61, 63, 69, 75, 87, 96, 109, 114, 135, 141, 154, 161, 170, 217, 228, 240]),
    'Resort': ([67, 104, 123, 124, 134, 138, 175, 184, 201, 255, 273, 275, 279, 283, 285],
              [11, 14, 76, 91, 114, 130, 132, 162, 163, 178, 191, 200, 207, 220, 226, 228, 246, 249, 281, 287]),
    'River': ([33, 76, 111, 161, 240, 292, 303, 307, 324, 331, 334, 347, 349, 375, 386],
             [38, 44, 64, 86, 125, 156, 208, 212, 229, 244, 255, 259, 264, 298, 313, 327, 344, 360, 366, 389]),
    'School': ([6, 12, 69, 94, 114, 137, 160, 164, 168, 171, 187, 191, 222, 235, 246],
              [17, 18, 19, 39, 84, 96, 108, 130, 132, 146, 152, 205, 207, 224, 227, 284, 286, 287, 290, 299]),
    'SparseResidential': ([16, 19, 34, 52, 76, 81, 112, 113, 134, 152, 160, 178, 235, 278, 285],
                         [1, 23, 27, 44, 54, 58, 62, 69, 78, 91, 92, 125, 126, 136, 141, 145, 179, 228, 241, 289]),
    'Square': ([45, 53, 68, 106, 112, 115, 117, 179, 205, 209, 250, 312, 317, 321, 330],
              [13, 24, 46, 59, 90, 91, 148, 165, 187, 193, 196, 202, 212, 232, 271, 282, 284, 304, 314, 329]),
    'Stadium': ([5, 6, 60, 69, 97, 117, 121, 143, 176, 205, 228, 242, 262, 284, 289],
               [4, 9, 23, 26, 28, 29, 31, 58, 61, 132, 135, 141, 145, 180, 183, 190, 197, 226, 266, 276]),
    'StorageTanks': ([25, 41, 62, 67, 99, 172, 223, 256, 260, 282, 301, 321, 341, 347, 349],
                    [31, 34, 57, 69, 73, 105, 110, 141, 189, 204, 206, 216, 234, 245, 249, 317, 320, 351, 358, 359]),
    'Viaduct': ([27, 73, 112, 125, 126, 150, 190, 226, 241, 250, 299, 329, 339, 400, 416],
               [16, 32, 62, 90, 103, 123, 161, 168, 183, 191, 213, 214, 240, 265, 266, 269, 343, 348, 386, 397]),
}

In [ ]:
def find_aid_root():
    # Папка, в якій лежать усі 30 папок-класів (беремо оригінальний AID, а не копії поділу)
    for base in (Path('data'), Path('../data'), Path('/kaggle/input')):
        if not base.exists():
            continue
        for p in sorted(base.rglob('Airport')):
            if p.is_dir() and all((p.parent / c).is_dir() for c in VAL_TEST) and 'AID_split' not in p.parts:
                return p.parent
    raise FileNotFoundError('Датасет AID не знайдено: завантажте його в data/ або підключіть як Kaggle input')


DATA_DIR = find_aid_root()
classes = sorted(VAL_TEST)
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

records = []
for c in classes:
    val_ids, test_ids = map(set, VAL_TEST[c])
    for f in sorted((DATA_DIR / c).glob('*.jpg')):
        n = int(f.stem.rsplit('_', 1)[1])
        split = 'val' if n in val_ids else 'test' if n in test_ids else 'train'
        records.append((str(f), c, class_to_idx[c], split))
df = pd.DataFrame(records, columns=['path', 'class', 'label', 'split'])

counts = pd.crosstab(df['class'], df['split'])
assert (counts['val'] == 15).all() and (counts['test'] == 20).all(), 'поділ не збігається з файлами датасету'
print('Папка з класами:', DATA_DIR)
print(f'Класів: {NUM_CLASSES}, зображень: {len(df)}', df['split'].value_counts().to_dict())

In [ ]:
class PathDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im.draft('RGB', (IMG_SIZE, IMG_SIZE))   # JPEG декодується одразу в 300×300 — швидше
            im = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)   # HWC -> CHW, uint8


IMAGES = torch.empty((len(df), 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
LABELS = torch.tensor(df['label'].values, dtype=torch.long)
pos = 0
for batch in tqdm(DataLoader(PathDataset(df['path']), batch_size=128, num_workers=NUM_WORKERS,
                             multiprocessing_context=MP_CONTEXT), desc='Кешування'):
    IMAGES[pos:pos + len(batch)] = batch
    pos += len(batch)

idx = {s: np.where(df['split'] == s)[0] for s in ('train', 'val', 'test')}
print(IMAGES.shape, f'{IMAGES.numel() / 1024**3:.2f} GB')

### Аугментація (як у завданнях 4–5)

У `__getitem__` для кожного навчального зображення випадково обирається одна з аугментацій: поворот на 90°/180°/270°,
горизонтальне чи вертикальне віддзеркалення або легке гаусове розмиття. Val/test — без аугментації.

In [ ]:
AUG_OPS = ('rot90', 'flip', 'blur')


def augment(x, op):
    # x: uint8-тензор 3×H×W
    if op == 'rot90':
        return torch.rot90(x, int(torch.randint(1, 4, ())), dims=(1, 2))
    if op == 'flip':
        return x.flip(2) if torch.rand(()) < 0.5 else x.flip(1)   # горизонтальне / вертикальне
    if op == 'blur':
        sigma = float(torch.empty(()).uniform_(*BLUR_SIGMA))
        return TF.gaussian_blur(x, kernel_size=[5, 5], sigma=[sigma, sigma])
    raise ValueError(op)


class AIDDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = np.asarray(indices)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        x = IMAGES[j]
        if self.augment:
            x = augment(x, AUG_OPS[int(torch.randint(len(AUG_OPS), ()))])
        return x, LABELS[j]


def make_loader(indices, augment=False, shuffle=False, seed=SEED):
    # Окремий генератор з тим самим seed -> однаковий порядок батчів (і аугментацій) у парних експериментах
    return DataLoader(AIDDataset(indices, augment), batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed), num_workers=NUM_WORKERS,
                      multiprocessing_context=MP_CONTEXT, persistent_workers=NUM_WORKERS > 0,
                      pin_memory=device.type == 'cuda')


train_loader = make_loader(idx['train'], shuffle=True)
train_loader_aug = make_loader(idx['train'], augment=True, shuffle=True)
train_eval_loader = make_loader(idx['train'])          # train без аугментації — для оцінки перенавчання
val_loader = make_loader(idx['val'])
test_loader = make_loader(idx['test'])

## 2. Спільні функції: моделі, навчання, оцінка

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1) * 255
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1) * 255


def to_input(x):
    return (x.to(device, non_blocking=True).float() - IMAGENET_MEAN) / IMAGENET_STD


def build_resnet18(pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def freeze_backbone(model, trainable=('fc',)):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(trainable)


def set_train_mode(model):
    # train() лише для шарів, що навчаються; заморожені BatchNorm лишаються в eval()
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()


def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    logits, labels = [], []
    for x, y in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            logits.append(model(to_input(x)).float().cpu())
        labels.append(y)
    return torch.cat(logits), torch.cat(labels)


def metrics(logits, labels):
    pred = logits.argmax(1).numpy()
    return {'loss': F.cross_entropy(logits, labels).item(),
            'acc': accuracy_score(labels, pred),
            'f1': f1_score(labels, pred, average='macro')}


def fit(model, train_loader, name, epochs=EPOCHS_ST, lr=LR_ST, loss_fn=None):
    # Навчання учня з нуля: AdamW + OneCycle (крок scheduler-а — на кожному батчі).
    # loss_fn(logits, x, y) — за замовчуванням cross-entropy; для дистиляції передаємо KD-втрату.
    loss_fn = loss_fn or (lambda logits, x, y: F.cross_entropy(logits, y))
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, epochs=epochs,
                                                    steps_per_epoch=len(train_loader), pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    hist = {k: [] for k in ('train_loss', 'train_acc', 'val_loss', 'val_acc', 'val_f1')}
    best_acc, best_state, best_epoch = -1, None, 0
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        set_train_mode(model)
        tot_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = to_input(x), y.to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = loss_fn(logits, x, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tot_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)

        val = metrics(*predict(model, val_loader))
        for k, v in (('train_loss', tot_loss / n), ('train_acc', correct / n),
                     ('val_loss', val['loss']), ('val_acc', val['acc']), ('val_f1', val['f1'])):
            hist[k].append(v)
        if val['acc'] > best_acc:   # вибір епохи — лише за validation
            best_acc, best_epoch = val['acc'], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f'[{name}] epoch {epoch:2d}/{epochs}  train loss {tot_loss / n:.4f} acc {correct / n:.4f} | '
              f'val loss {val["loss"]:.4f} acc {val["acc"]:.4f} f1 {val["f1"]:.4f}  ({time.time() - t0:.0f} s)')

    model.load_state_dict(best_state)
    torch.save(best_state, OUT_DIR / f'{name}.pt')
    hist.update(best_epoch=best_epoch, train_time=time.time() - t0)
    print(f'[{name}] найкраща епоха за val acc: {best_epoch} ({best_acc:.4f}), ваги -> {OUT_DIR / name}.pt')
    return hist


results, histories, test_preds = {}, {}, {}


def evaluate_model(model, name, hist):
    row = {'trainable_params': count_params(model, trainable_only=True), 'best_epoch': hist['best_epoch'],
           'train_time_s': round(hist['train_time'])}
    for split, loader in (('train', train_eval_loader), ('val', val_loader), ('test', test_loader)):
        logits, labels = predict(model, loader)
        m = metrics(logits, labels)
        row[f'{split}_acc'], row[f'{split}_f1'] = m['acc'], m['f1']
        if split == 'test':
            test_preds[name] = logits.argmax(1).numpy()
    row['train-val gap'] = row['train_acc'] - row['val_acc']
    results[name], histories[name] = row, hist
    return pd.DataFrame(row, index=[name]).round(4)

### Моделі з попередніх завдань

Учитель — модель із **найвищою val accuracy** серед exp3, exp4, exp5 (за рівності — вища val macro-F1).
Test для вибору не використовується.

In [ ]:
PREV_BASES = (Path('../steps3_4/out'), Path('../step5/out'), Path('steps3_4/out'), Path('step5/out'),
              Path('/kaggle/input'))


def find_prev(filename):
    for base in PREV_BASES:
        if base.exists():
            hits = sorted(base.rglob(filename))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'{filename} не знайдено — підключіть вихід попередніх кернелів')


def load_resnet(name):
    model = build_resnet18(pretrained=False)
    model.load_state_dict(torch.load(find_prev(f'{name}.pt'), map_location='cpu'))
    for p in model.parameters():
        p.requires_grad = False
    return model.to(device).eval()


prev_results = pd.read_csv(find_prev('results_step5.csv'), index_col=0)   # exp3, exp4, exp5
prev_hist = {**json.load(open(find_prev('histories_steps3_4.json'))), **json.load(open(find_prev('histories_step5.json')))}

TEACHER = prev_results.sort_values(['val_acc', 'val_f1'], ascending=False).index[0]
print('Учитель (найкращий за validation):', TEACHER)
display(prev_results[['trainable_params', 'val_acc', 'val_f1', 'test_acc', 'test_f1']].round(4))

results, histories, test_preds = {}, {}, {}
MODELS = {name: load_resnet(name) for name in prev_results.index}
for name, model in MODELS.items():
    evaluate_model(model, name, prev_hist[name])
    results[name]['trainable_params'] = int(prev_results.loc[name, 'trainable_params'])   # під час навчання, а не зараз
teacher = MODELS[TEACHER]
pd.DataFrame(results).T[['val_acc', 'val_f1', 'test_acc', 'test_f1']].round(4)

## 3. Завдання 6: компактна CNN-учень (звичайне навчання)

**Архітектура** — проста VGG-подібна мережа, що навчається **з нуля** (без ImageNet-ваг), тож ефект дистиляції
в завданні 7 «чистий» — учень не отримує знань з ImageNet іншим шляхом:

```
вхід 3×224×224
Conv3×3(stride 2)-BN-ReLU, Conv3×3-BN-ReLU, 32 канали  → MaxPool   -> 56×56
[Conv3×3-BN-ReLU] ×2, 64 канали                         → MaxPool   -> 28×28
[Conv3×3-BN-ReLU] ×2, 128 каналів                       → MaxPool   -> 14×14
[Conv3×3-BN-ReLU] ×2, 256 каналів                                   -> 14×14
GlobalAvgPool → Dropout(0.3) → Linear(256 → 30)
```

**Навчання:** AdamW (weight decay 5e-4), OneCycle з max LR 3e-3, 40 епох, та сама аугментація, що й у вчителя.
Початкові ваги учня зберігаються (`student_init.pt`) — учень із дистиляцією стартує з **тих самих** ваг.

In [ ]:
def conv_bn(cin, cout, stride=1):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn(3, 32, stride=2), conv_bn(32, 32), nn.MaxPool2d(2),
            conv_bn(32, 64), conv_bn(64, 64), nn.MaxPool2d(2),
            conv_bn(64, 128), conv_bn(128, 128), nn.MaxPool2d(2),
            conv_bn(128, 256), conv_bn(256, 256),
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


set_seed()
student_init = SmallCNN().state_dict()
torch.save(student_init, OUT_DIR / 'student_init.pt')

print(f'Параметрів учня: {count_params(SmallCNN()):,}; учителя ({TEACHER}): {count_params(teacher):,} '
      f'-> у {count_params(teacher) / count_params(SmallCNN()):.1f} раза менше')

In [ ]:
def new_student():
    model = SmallCNN()
    model.load_state_dict(student_init)
    return model.to(device)


set_seed()
student = new_student()
hist_st = fit(student, train_loader_aug, 'exp6_student')
MODELS['exp6_student'] = student
evaluate_model(student, 'exp6_student', hist_st)

## 4. Завдання 7: учень із дистиляцією знань

**Функція втрат** (Hinton et al., 2015):

$$L = \alpha \cdot T^2 \cdot \mathrm{KL}\big(\mathrm{softmax}(z_t/T)\,\|\,\mathrm{softmax}(z_s/T)\big) + (1-\alpha)\cdot \mathrm{CE}(z_s, y)$$

* $z_t$, $z_s$ — логіти вчителя та учня;
* **T — температура** (за замовчуванням **T = 4**): «розм'якшує» розподіл учителя, щоб учень бачив, які класи вчитель
  вважає схожими (напр., `School` ↔ `Commercial`), а не лише правильну відповідь;
* **α — вага дистиляційної складової** (за замовчуванням **α = 0.7**); $1-\alpha$ — вага звичайної CE з мітками;
* множник $T^2$ компенсує зменшення градієнтів через ділення логітів на T.

**Чесне порівняння з exp6:** ті самі `student_init.pt`, епохи, оптимізатор, scheduler, аугментація і порядок батчів.
Відрізняється **лише** функція втрат. Учитель — у `eval()`, без градієнтів, отримує **той самий аугментований батч**.
Якщо в `KD_CONFIGS` кілька пар (T, α), кожна навчається окремо, а в exp7 потрапляє краща **за validation**.

In [ ]:
def make_kd_loss(teacher, T, alpha):
    teacher.eval()

    def kd_loss(logits, x, y):
        with torch.no_grad():
            t_logits = teacher(x)
        kd = F.kl_div(F.log_softmax(logits.float() / T, dim=1), F.softmax(t_logits.float() / T, dim=1),
                      reduction='batchmean') * T * T
        return alpha * kd + (1 - alpha) * F.cross_entropy(logits, y)

    return kd_loss


kd_runs = {}
for T, alpha in KD_CONFIGS:
    set_seed()
    model = new_student()
    hist = fit(model, train_loader_aug, f'kd_T{T:g}_a{alpha:g}', loss_fn=make_kd_loss(teacher, T, alpha))
    kd_runs[(T, alpha)] = (model, hist, max(hist['val_acc']))

(KD_T, KD_ALPHA), (student_kd, hist_kd, _) = max(kd_runs.items(), key=lambda kv: kv[1][2])
if len(kd_runs) > 1:
    display(pd.DataFrame([(T, a, v[2]) for (T, a), v in kd_runs.items()], columns=['T', 'alpha', 'best val acc']))
print(f'Обрано T = {KD_T}, α = {KD_ALPHA}')
torch.save(student_kd.state_dict(), OUT_DIR / 'exp7_student_kd.pt')
MODELS['exp7_student_kd'] = student_kd
hist_kd = {**hist_kd, 'T': KD_T, 'alpha': KD_ALPHA}
evaluate_model(student_kd, 'exp7_student_kd', hist_kd)

In [ ]:
def plot_histories(hists, title):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for (name, h), color in zip(hists.items(), plt.rcParams['axes.prop_cycle'].by_key()['color']):
        ep = range(1, len(h['train_loss']) + 1)
        axes[0].plot(ep, h['train_loss'], color=color, ls='--', label=f'{name} train')
        axes[0].plot(ep, h['val_loss'], color=color, label=f'{name} val')
        axes[1].plot(ep, h['train_acc'], color=color, ls='--', label=f'{name} train')
        axes[1].plot(ep, h['val_acc'], color=color, label=f'{name} val')
        axes[2].plot(ep, h['val_f1'], color=color, label=f'{name} val')
        for ax in axes:
            ax.axvline(h['best_epoch'], color=color, lw=0.8, ls=':')
    for ax, t in zip(axes, ('Loss (для exp7 train loss — KD-втрата, не порівнювати напряму)', 'Accuracy', 'Macro-F1')):
        ax.set_title(t, fontsize=10); ax.set_xlabel('Епоха'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_histories({'exp6_student': hist_st, 'exp7_student_kd': hist_kd}, 'Учень: звичайне навчання vs дистиляція')


def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.state_dict().values()) / 1024**2


trio = [TEACHER, 'exp6_student', 'exp7_student_kd']
cmp = pd.DataFrame(results).T.loc[trio, ['val_acc', 'val_f1', 'test_acc', 'test_f1', 'train-val gap']]
cmp.insert(0, 'params', [count_params(MODELS[n]) for n in trio])
cmp.insert(1, 'size_MB', [model_size_mb(MODELS[n]) for n in trio])
print(f'Дистиляція: T = {KD_T}, α = {KD_ALPHA}')
cmp.round(4)

**Висновок до завдань 6–7** _(заповнити після запуску)_: на скільки учень поступається вчителю за accuracy / macro-F1
при у ~9.5 раза меншій кількості параметрів; чи покращила дистиляція учня порівняно зі звичайним навчанням
(з тими самими початковими вагами й бюджетом); як змінився розрив train–val (дистиляція — теж регуляризація:
м'які цілі вчителя менш «впевнені», ніж one-hot мітки).

## 5. Завдання 8: аналіз помилок учителя та учнів

### Метрики для кожного класу (test)

* **precision** — частка зображень, віднесених до класу, які справді йому належать;
* **recall** — частка зображень класу, розпізнаних правильно, тобто **accuracy класу**;
* **accuracy (one-vs-rest)** — (TP + TN) / N для задачі «клас vs решта»; завжди близька до 1 через велику кількість TN.

20 тестових зображень на клас: зміна recall на 0.05 = одне зображення.

In [ ]:
y_test = LABELS[idx['test']].numpy()
test_idx = idx['test']


def per_class_metrics(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    tn = cm.sum() - tp - fp - fn
    out = pd.DataFrame({'precision': p, 'recall (= accuracy класу)': r, 'f1': f,
                        'accuracy (one-vs-rest)': (tp + tn) / cm.sum()}, index=classes)
    out.loc['macro avg'] = out.mean()
    return out


pc = {name: per_class_metrics(y_test, test_preds[name]) for name in trio}
per_class = pd.concat({name: pc[name] for name in trio}, axis=1).sort_values((TEACHER, 'f1'))
per_class.to_csv(OUT_DIR / 'per_class_teacher_students.csv')

(per_class.style.format('{:.3f}')
 .background_gradient(cmap='RdYlGn', vmin=0.4, vmax=1.0,
                      subset=[c for c in per_class.columns if c[1] != 'accuracy (one-vs-rest)']))

### Матриці помилок

In [ ]:
def plot_confusion(name):
    cm = confusion_matrix(y_test, test_preds[name], labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.imshow(cm / cm.sum(1, keepdims=True), cmap='Blues', vmin=0, vmax=1)
    for i, j in zip(*np.nonzero(cm)):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                color='white' if i == j and cm[i, j] > 12 else ('black' if i == j else 'crimson'))
    ax.set_xticks(range(NUM_CLASSES), classes, rotation=90, fontsize=8)
    ax.set_yticks(range(NUM_CLASSES), classes, fontsize=8)
    ax.set_xlabel('Прогноз'); ax.set_ylabel('Справжній клас')
    ax.set_title(f'Матриця помилок: {name} (test acc {results[name]["test_acc"]:.3f}); червоні — помилки')
    plt.tight_layout()
    plt.show()


for name in trio:
    plot_confusion(name)

In [ ]:
# Найчастіші пари «справжній клас → прогноз» для кожної моделі
def top_confusions(name, k=10):
    cm = confusion_matrix(y_test, test_preds[name], labels=range(NUM_CLASSES))
    np.fill_diagonal(cm, 0)
    pairs = [(classes[i], classes[j], cm[i, j]) for i, j in zip(*np.nonzero(cm))]
    pairs = sorted(pairs, key=lambda t: -t[2])[:k]
    return pd.DataFrame([f'{a} → {b} ({n})' for a, b, n in pairs], columns=[name])


pd.concat([top_confusions(n) for n in trio], axis=1)

### Приклади неправильних прогнозів

In [ ]:
def show_images(items, n_cols=6, size=2.6, suptitle=None):
    # items: список (позиція в test, заголовок)
    n_rows = int(np.ceil(len(items) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, (size + 0.4) * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (pos, title) in zip(axes.flat, items):
        ax.imshow(IMAGES[test_idx[pos]].permute(1, 2, 0).numpy())
        ax.set_title(title, fontsize=8)
    if suptitle:
        plt.suptitle(suptitle, y=1.0)
    plt.tight_layout()
    plt.show()


test_probs = {name: F.softmax(predict(MODELS[name], test_loader)[0], 1).numpy() for name in MODELS}
rng = np.random.default_rng(SEED)
for name in trio:
    wrong = np.where(test_preds[name] != y_test)[0]
    pick = rng.choice(wrong, min(12, len(wrong)), replace=False)
    show_images([(i, f'{classes[y_test[i]]}\n→ {classes[test_preds[name][i]]} ({test_probs[name][i].max():.2f})')
                 for i in pick], suptitle=f'{name}: неправильні прогнози (справжній → прогноз (ймовірність)), '
                                          f'усього помилок {len(wrong)} з {len(y_test)}')

### Grad-CAM

Grad-CAM показує, які ділянки зображення найбільше вплинули на рішення: беремо активації **останнього згорткового
шару** (`layer4` у ResNet18, останній `conv_bn` в учня), зважуємо кожен канал середнім градієнтом логіта обраного
класу за цим каналом, підсумовуємо, застосовуємо ReLU і розтягуємо до розміру зображення.
Карту будуємо для **передбаченого** класу кожної моделі (тобто «чому модель так вирішила»).

Нижче — зображення трьох груп: усі моделі праві; учитель помилився; учитель правий, а учні помилились.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts = model, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o))

    def __call__(self, x_uint8):
        self.model.eval()
        x = to_input(x_uint8[None]).requires_grad_(True)   # requires_grad, щоб градієнт ішов і через заморожені шари
        logits = self.model(x)
        cls = logits.argmax(1).item()
        grads = torch.autograd.grad(logits[0, cls], self.acts)[0]
        cam = F.relu((grads.mean((2, 3), keepdim=True) * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls


cam_layers = {TEACHER: MODELS[TEACHER].layer4, 'exp6_student': MODELS['exp6_student'].features[-1],
              'exp7_student_kd': MODELS['exp7_student_kd'].features[-1]}
cams = {name: GradCAM(MODELS[name], layer) for name, layer in cam_layers.items()}

ok = {n: test_preds[n] == y_test for n in trio}
groups = {
    'усі праві': np.where(ok[TEACHER] & ok['exp6_student'] & ok['exp7_student_kd'])[0],
    'учитель помилився': np.where(~ok[TEACHER])[0],
    'учитель правий, учні — ні': np.where(ok[TEACHER] & ~ok['exp6_student'] & ~ok['exp7_student_kd'])[0],
}
rng = np.random.default_rng(SEED + 1)
cam_items = [(g, i) for g, pool in groups.items() for i in rng.choice(pool, min(3, len(pool)), replace=False)]

fig, axes = plt.subplots(len(cam_items), 4, figsize=(12, 3.1 * len(cam_items)), squeeze=False)
for r, (group, i) in enumerate(cam_items):
    img = IMAGES[test_idx[i]]
    axes[r, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[r, 0].set_title(f'[{group}]\nсправжній: {classes[y_test[i]]}', fontsize=9)
    for c, name in enumerate(trio, start=1):
        cam, cls = cams[name](img)
        axes[r, c].imshow(img.permute(1, 2, 0).numpy())
        axes[r, c].imshow(cam, cmap='jet', alpha=0.45)
        axes[r, c].set_title(f'{name}\n→ {classes[cls]}', fontsize=9, color='green' if cls == y_test[i] else 'red')
    for ax in axes[r]:
        ax.axis('off')
plt.tight_layout()
plt.show()

**Спостереження** _(заповнити після запуску)_:
* на правильних прогнозах учитель зазвичай дивиться на характерні об'єкти класу (злітна смуга, трибуни стадіону,
  резервуари, вода біля берега), а не на фон;
* помилки вчителя часто — на неоднозначних сценах, де характерний об'єкт малий або є ознаки кількох класів
  (школа з парковкою й житловими будинками навколо, центр міста ↔ площа ↔ комерційна зона);
* карти учнів грубіші (останній шар 14×14 проти 7×7 у ResNet18, але слабші ознаки) і частіше «розмазані» по текстурі;
  чи стала карта учня з дистиляцією більш схожою на карту вчителя?

## 6. Завдання 9: покращення та погіршення по класах (exp4 і exp7)

* **exp4 vs exp3** — вплив **аугментації** (заморожена ResNet18);
* **exp7 vs exp6** — вплив **дистиляції** (той самий учень, ті самі початкові ваги).

Порівнюємо F1 і recall по класах на test та показуємо виправлені помилки (було неправильно → стало правильно)
і нові помилки (було правильно → стало неправильно).

In [ ]:
PAIRS = [('exp3_frozen', 'exp4_frozen_aug', 'аугментація'), ('exp6_student', 'exp7_student_kd', 'дистиляція')]

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, (before, after, label) in zip(axes, PAIRS):
    m0, m1 = per_class_metrics(y_test, test_preds[before]), per_class_metrics(y_test, test_preds[after])
    d = pd.DataFrame({'ΔF1': m1['f1'] - m0['f1'],
                      'Δrecall': m1['recall (= accuracy класу)'] - m0['recall (= accuracy класу)']})
    d = d.drop(index='macro avg').sort_values('ΔF1')
    y = np.arange(len(d))
    ax.barh(y - 0.2, d['ΔF1'], height=0.4, color=np.where(d['ΔF1'] >= 0, 'seagreen', 'indianred'), label='ΔF1')
    ax.barh(y + 0.2, d['Δrecall'], height=0.4, color='lightgray', label='Δrecall')
    ax.set_yticks(y, d.index, fontsize=9)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{label}: {after} − {before} (test)')
    ax.legend(loc='lower right')
    print(f'{label}: найбільше покращення —', ', '.join(f'{c} ({v:+.2f})' for c, v in d['ΔF1'].tail(3)[::-1].items()),
          '| найбільше погіршення —', ', '.join(f'{c} ({v:+.2f})' for c, v in d['ΔF1'].head(3).items()))
plt.tight_layout()
plt.show()

In [ ]:
for before, after, label in PAIRS:
    fixed = np.where((test_preds[before] != y_test) & (test_preds[after] == y_test))[0]
    broken = np.where((test_preds[before] == y_test) & (test_preds[after] != y_test))[0]
    print(f'{label} ({before} → {after}): виправлено {len(fixed)}, нових помилок {len(broken)}, '
          f'чистий виграш {len(fixed) - len(broken):+d} з {len(y_test)}')
    rng = np.random.default_rng(SEED)
    for kind, pool in (('виправлені помилки', fixed), ('нові помилки', broken)):
        if len(pool) == 0:
            continue
        pick = rng.choice(pool, min(6, len(pool)), replace=False)
        show_images([(i, f'{classes[y_test[i]]}\nбуло: {classes[test_preds[before][i]]}\n'
                         f'стало: {classes[test_preds[after][i]]}') for i in pick],
                    suptitle=f'{label}: {kind}')

**Висновок до завдання 9** _(заповнити після запуску)_: які класи найбільше виграли від аугментації та від
дистиляції; чи збігаються ці класи (напр., «складні» міські класи); скільки помилок виправлено і скільки нових
з'явилось — зміна способу навчання не лише виправляє помилки, а й «перетасовує» їх.

## 7. Завдання 10: зведені результати

### Стійкість до спотворень

Оцінюємо всі моделі на **test** зі спотвореннями, яких не було в навчанні (або були лише слабкими):
гаусів шум, сильне розмиття, затемнення, зниження контрасту, сильне JPEG-стиснення, закриття частини зображення
(«хмара»), а також поворот на 90° (до нього стійкі лише моделі, навчені з аугментацією).
**Стійкість** — середня accuracy на спотворених версіях і середнє падіння відносно чистого test.

In [ ]:
def jpeg(x_uint8, quality):
    out = []
    for img in x_uint8:
        buf = io.BytesIO()
        Image.fromarray(img.permute(1, 2, 0).numpy()).save(buf, format='JPEG', quality=quality)
        out.append(torch.from_numpy(np.asarray(Image.open(buf)).copy()).permute(2, 0, 1))
    return torch.stack(out)


def perturb(X, kind):
    g = torch.Generator().manual_seed(SEED)
    x = X.float() / 255
    if kind == 'шум σ=0.08':
        x = x + torch.randn(x.shape, generator=g) * 0.08
    elif kind == 'розмиття σ=2':
        x = TF.gaussian_blur(x, kernel_size=[9, 9], sigma=[2.0, 2.0])
    elif kind == 'затемнення ×0.6':
        x = x * 0.6
    elif kind == 'контраст ×0.5':
        m = x.mean((1, 2, 3), keepdim=True)
        x = m + (x - m) * 0.5
    elif kind == 'JPEG q=10':
        return jpeg(X, 10)
    elif kind == 'закриття 40%':
        s = int(IMG_SIZE * 0.4)
        x = x.clone()
        for k in range(len(x)):
            i, j = torch.randint(0, IMG_SIZE - s, (2,), generator=g).tolist()
            x[k, :, i:i + s, j:j + s] = 0.85   # світла пляма, як хмара
    elif kind == 'поворот 90°':
        x = torch.rot90(x, 1, dims=(2, 3))
    return (x.clamp(0, 1) * 255).round().to(torch.uint8)


PERTURBATIONS = ['чистий', 'шум σ=0.08', 'розмиття σ=2', 'затемнення ×0.6', 'контраст ×0.5', 'JPEG q=10',
                 'закриття 40%', 'поворот 90°']
X_test, Y_test = IMAGES[test_idx], LABELS[test_idx]
robust = pd.DataFrame(index=list(MODELS), columns=PERTURBATIONS, dtype=float)
examples = {}
for kind in tqdm(PERTURBATIONS, desc='Спотворення'):
    Xp = perturb(X_test, kind)
    examples[kind] = Xp[0]
    loader = DataLoader(TensorDataset(Xp, Y_test), batch_size=BATCH_SIZE)
    for name, model in MODELS.items():
        robust.loc[name, kind] = metrics(*predict(model, loader))['acc']

fig, axes = plt.subplots(1, len(PERTURBATIONS), figsize=(2.2 * len(PERTURBATIONS), 2.6))
for ax, kind in zip(axes, PERTURBATIONS):
    ax.imshow(examples[kind].permute(1, 2, 0).numpy()); ax.set_title(kind, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.show()

robust['середня (спотворені)'] = robust[PERTURBATIONS[1:]].mean(1)
robust['середнє падіння'] = robust['чистий'] - robust['середня (спотворені)']
robust.to_csv(OUT_DIR / 'robustness.csv')
robust.style.format('{:.3f}').background_gradient(cmap='RdYlGn', vmin=0.2, vmax=1.0, subset=PERTURBATIONS + ['середня (спотворені)'])

### Розмір і швидкість моделей

In [ ]:
@torch.no_grad()
def gpu_throughput(model, bs=64, n=20):
    if device.type != 'cuda':
        return np.nan
    model.eval()
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.autocast('cuda', dtype=torch.float16):
        for _ in range(3):
            model(x)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            model(x)
        torch.cuda.synchronize()
    return bs * n / (time.time() - t0)


@torch.no_grad()
def cpu_latency_ms(model, n=20):
    m = copy.deepcopy(model).cpu().float().eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    for _ in range(3):
        m(x)
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        m(x)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


speed = pd.DataFrame({name: {'params': count_params(m), 'size_MB': model_size_mb(m),
                             'GPU img/s': gpu_throughput(m), 'CPU latency, ms': cpu_latency_ms(m)}
                      for name, m in MODELS.items()}).T
speed.round(1)

### Зведена таблиця

In [ ]:
DESCR = {'exp3_frozen': 'ResNet18, заморожена основа', 'exp4_frozen_aug': 'ResNet18, заморожена основа + аугм.',
         'exp5_finetune': 'ResNet18, fine-tuning layer4 + аугм.', 'exp6_student': 'SmallCNN з нуля + аугм.',
         'exp7_student_kd': f'SmallCNN + дистиляція (T={KD_T:g}, α={KD_ALPHA:g})'}
res = pd.DataFrame(results).T
final = pd.DataFrame({
    'модель': [DESCR[n] for n in MODELS],
    'параметрів': speed['params'].astype(int),
    'розмір, МБ': speed['size_MB'],
    'параметрів, що навчались': res['trainable_params'].astype(int),
    'val acc': res['val_acc'], 'val F1': res['val_f1'],
    'test acc': res['test_acc'], 'test F1': res['test_f1'],
    'train-val gap': res['train-val gap'],
    'стійкість (acc на спотв.)': robust['середня (спотворені)'],
    'падіння від спотв.': robust['середнє падіння'],
    'GPU img/s': speed['GPU img/s'], 'CPU latency, ms': speed['CPU latency, ms'],
}, index=list(MODELS))
final.to_csv(OUT_DIR / 'results_all.csv')

best_acc = final['test acc'].idxmax()
smallest = ', '.join(final.index[final['параметрів'] == final['параметрів'].min()])
most_robust = final['стійкість (acc на спотв.)'].idxmax()
print(f'Найточніша (test acc):       {best_acc} — {final.loc[best_acc, "test acc"]:.3f}')
print(f'Найкомпактніші (параметри):  {smallest} — {final["параметрів"].min():,} '
      f'({final["розмір, МБ"].min():.1f} МБ)')
print(f'Найстійкіша (acc на спотв.): {most_robust} — {final.loc[most_robust, "стійкість (acc на спотв.)"]:.3f} '
      f'(падіння {final.loc[most_robust, "падіння від спотв."]:.3f})')

np.savez(OUT_DIR / 'test_preds_all.npz', y_test=y_test, test_idx=test_idx, **test_preds)
json.dump(histories, open(OUT_DIR / 'histories_steps6_10.json', 'w'))
final.round(3)

## 8. Висновки

_(заповнити після запуску, спираючись на зведену таблицю)_

* **Найточніша модель** — ... (test accuracy / macro-F1). Fine-tuning останнього блоку дав найбільший приріст серед
  усіх змін: модель адаптує високорівневі ознаки ImageNet до аерознімків.
* **Найкомпактніша** — учень SmallCNN (~1.2 млн параметрів, ~4.5 МБ, у ~9.5 раза менше за ResNet18), швидший на CPU.
  Ціна компактності — ... п.п. accuracy. Дистиляція скоротила / не скоротила цей розрив на ... п.п.
* **Найстійкіша** — ... (найвища accuracy на спотворених зображеннях / найменше падіння). Моделі з аугментацією
  стійкі до поворотів; до шуму та JPEG-артефактів ... ; учні, навчені з нуля, ... .
* **Аугментація** зі замороженою основою майже не змінила точність (навчався лише лінійний шар), але зробила модель
  інваріантною до поворотів; у поєднанні з fine-tuning та для учня з нуля — ... .
* **Типові помилки** всіх моделей — між семантично близькими міськими класами (`School`, `Commercial`, `Center`,
  `Square`, `Resort`) та між житловими зонами різної щільності; Grad-CAM показує, що ... .